<a href="https://colab.research.google.com/github/Emily-als/worklens-llm/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training

Runs and evaluates one model per run, driven by a YAML config from the repo's `configs/` folder.
`run.mode` in the config picks what happens:

| Config | `zero_shot` | `few_shot` | `finetune` |
|---|---|---|---|
| `qwen3-1.7b` | prompt only | k examples per class in the prompt | LoRA (fp16) |
| `phi-4-mini` | prompt only | k examples per class in the prompt | QLoRA (4-bit) |
| `deberta-v3-base` | NLI model DeBERTa-v3-large-zeroshot | NLI model fine-tuned on k reviews per class | full fine-tune of DeBERTa-v3-base |
| `roberta-base` | NLI model RoBERTa-large-MNLI | NLI model fine-tuned on k reviews per class | full fine-tune of RoBERTa-base |

**How to use**

1. Runtime > Change runtime type > **T4 GPU**.
2. Edit the config in the repo (`configs/<model>.yaml`): `run.mode`, `run.condition`, `run.scale`, `run.seed`, hyperparameters, or a `sweep`
   (e.g. `sweep: {mode: [zero_shot, few_shot]}` runs both baselines). Push it.
3. Set `CONFIG_NAME` below and **Run all**.

Metrics: **macro-F1** (main metric), **weighted F1** and **accuracy**, per aspect and averaged over the 3 aspects.

Every run writes to `RUNS/<run_id>/` on Drive: config snapshot, final model or LoRA adapter, predictions per test set, and `metrics.csv`.
Scores are also appended to `RUNS/results.csv`. A run whose `metrics.csv` already exists is skipped; an unfinished run resumes from its last checkpoint.

Each review is classified **one aspect at a time** (work-life balance, compensation & benefits, senior management) into Low / Medium / High.
LLMs answer with one word and are scored by next-token probability of the three labels; encoders get `(aspect, review)` as a sentence pair.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Emily-als/worklens-llm.git"
REPO_BRANCH = "feature/model-training"   # switch to "main" once the branch is merged
CONFIG_NAME = "qwen3-1.7b"               # qwen3-1.7b | phi-4-mini | deberta-v3-base | roberta-base

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes sentencepiece

## 2. Setup: Drive, repo, config

In [ ]:
import copy, gc, itertools, math, shutil, subprocess, time
from tqdm.auto import tqdm
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from google.colab import drive

drive.mount("/content/drive")

# Get the configs from the repo (clone once, then always sync to the latest commit of REPO_BRANCH)
REPO_DIR = Path("/content/worklens-llm")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)


def deep_merge(base, extra):
    out = copy.deepcopy(base)
    for key, value in extra.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = value
    return out


def load_config(name):
    base = yaml.safe_load((REPO_DIR / "configs" / "base.yaml").read_text())
    model = yaml.safe_load((REPO_DIR / "configs" / f"{name}.yaml").read_text())
    return deep_merge(base, model)


CFG = load_config(CONFIG_NAME)
print(yaml.safe_dump(CFG, sort_keys=False))

DATA_DIR = Path(CFG["paths"]["data_dir"])
RUNS_DIR = Path(CFG["paths"]["runs_dir"])
RESULTS_CSV = Path(CFG["paths"]["results_csv"])
RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
assert (DATA_DIR / "train.jsonl").exists(), f"train.jsonl not found in {DATA_DIR}: fix paths.data_dir in configs/base.yaml"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Data

Fine-tuning data = real reviews for the chosen `scale` + the augmentation file of the chosen `condition`.
Few-shot examples are always real reviews, drawn from the `scale` file. Zero-shot uses no training data.

| scale | real file | augmentation files |
|---|---|---|
| `full` | `train.jsonl` (20k) | `aug_*.jsonl` (10k) |
| `low` | `train_2k.jsonl` | `aug_*_1k.jsonl` (1k) |
| `tune` | `train_10k.jsonl` | none (C1 only) |

In [ ]:
ASPECTS = {
    "work_life_balance": "work-life balance",
    "compensation_benefits": "compensation and benefits",
    "senior_management": "senior management",
}
LABELS = ["Low", "Medium", "High"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

TRAIN_FILES = {"full": "train", "low": "train_2k", "tune": "train_10k"}
AUG_FILES = {
    "C1": None,
    "C2": "aug_noise",
    "C3": "aug_oversample_mixed",
    "C4": "aug_synthetic",
    "C5": "aug_mixed_rewrites",
}


def load_split(name):
    df = pd.read_json(DATA_DIR / f"{name}.jsonl", lines=True, dtype={"id": str})
    df["text"] = df["text"].str.replace("\r\n", "\n")
    return df


def to_pairs(df):
    """One {text, aspect, label} item per review and aspect."""
    return [
        {"text": text, "aspect": aspect, "label": label}
        for aspect in ASPECTS
        for text, label in zip(df["text"], df[aspect])
    ]


def training_pairs(run):
    """finetune: real + augmentation pairs. few_shot: k real reviews per class, sampled separately per aspect."""
    df = load_split(TRAIN_FILES[run["scale"]])
    if run["mode"] == "few_shot":
        pairs = []
        for aspect in ASPECTS:
            part = df.groupby(aspect, group_keys=False).sample(n=run["k_shot"], random_state=run["seed"])
            pairs += [p for p in to_pairs(part) if p["aspect"] == aspect]
        return pairs

    assert run["scale"] != "tune" or run["condition"] == "C1", "tune runs use real data only: set condition C1"
    aug = AUG_FILES[run["condition"]]
    if aug:
        df = pd.concat([df, load_split(aug + ("_1k" if run["scale"] == "low" else ""))], ignore_index=True)
    return to_pairs(df)


def make_prompt(review, aspect, shots=()):
    """shots: (review, label) examples shown before the review (LLM few-shot)."""
    examples = "".join(f"Review: {text}\nAnswer: {label}\n\n" for text, label in shots)
    return (
        f"Rate how satisfied the employee is with {ASPECTS[aspect]}, based on the review.\n"
        "Answer with exactly one word: Low, Medium, or High.\n\n"
        f"{examples}Review: {review}\nAnswer:"
    )


print({name: len(load_split(name)) for name in CFG["eval"]["sets"]})

## 4. Models

Every model class has `predict(texts, aspect, batch_size)`. `LLM` and `Encoder` also have `train(pairs, val_pairs, run_dir, cfg, seed)`;
`LLM.set_shots()` adds in-context examples for few-shot prompting; `NLIZeroShot` needs no training.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


def last_checkpoint(run_dir):
    return any(run_dir.glob("checkpoint-*"))


def remove_checkpoints(run_dir):
    for ckpt in run_dir.glob("checkpoint-*"):
        shutil.rmtree(ckpt)


def common_args(run_dir, t, seed, n_train):
    accum = t.get("grad_accum", 1)
    total_steps = math.ceil(n_train / (t["batch_size"] * accum)) * t["epochs"]
    return dict(
        output_dir=str(run_dir),
        per_device_train_batch_size=t["batch_size"],
        gradient_accumulation_steps=accum,
        num_train_epochs=t["epochs"],
        learning_rate=float(t["lr"]),
        warmup_steps=int(total_steps * t["warmup_ratio"]),
        weight_decay=t["weight_decay"],
        fp16=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        seed=seed,
    )


class LLM:
    """Causal LM + LoRA / QLoRA. Trained to answer the label word; predicts by comparing the 3 label-token logits."""

    def __init__(self, cfg):
        m = cfg["model"]
        self.tok = AutoTokenizer.from_pretrained(m["hf_id"])
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        quant = None
        if m["quantization"] == "4bit":
            quant = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        self.model = AutoModelForCausalLM.from_pretrained(
            m["hf_id"], dtype=torch.float16, quantization_config=quant, device_map={"": 0}
        )
        self.max_review_tokens = cfg["train"]["max_length"] - 128
        self.template_kwargs = {"enable_thinking": False} if "qwen3" in m["hf_id"].lower() else {}
        self.label_ids = [self.tok.encode(label, add_special_tokens=False)[0] for label in LABELS]
        assert len(set(self.label_ids)) == 3, f"labels share a first token: {self.label_ids}"
        self.shots = {}  # aspect -> [(review, label)], filled by set_shots() for few-shot prompting
        self.tokens_seen = 0  # input tokens processed by predict(), for the tokens/s speed metric

    def clip(self, text, max_tokens):
        ids = self.tok.encode(text, add_special_tokens=False)
        return text if len(ids) <= max_tokens else self.tok.decode(ids[:max_tokens])

    def set_shots(self, pairs, seed, max_tokens=150):
        """Few-shot prompting: the pairs become in-context examples, shuffled so classes are not grouped."""
        rng = np.random.default_rng(seed)
        for aspect in ASPECTS:
            shots = [(self.clip(p["text"], max_tokens), p["label"]) for p in pairs if p["aspect"] == aspect]
            rng.shuffle(shots)
            self.shots[aspect] = shots

    def prompt_ids(self, review, aspect):
        review = self.clip(review, self.max_review_tokens)
        text = self.tok.apply_chat_template(
            [{"role": "user", "content": make_prompt(review, aspect, self.shots.get(aspect, ()))}],
            tokenize=False,
            add_generation_prompt=True,
            **self.template_kwargs,
        )
        return self.tok.encode(text, add_special_tokens=False)  # the chat template already adds BOS if needed

    def collate(self, batch):
        width = max(len(item["input_ids"]) for item in batch)
        pad = lambda seq, value: seq + [value] * (width - len(seq))
        return {
            "input_ids": torch.tensor([pad(b["input_ids"], self.tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
        }

    def train(self, pairs, val_pairs, run_dir, cfg, seed):
        t, lora = cfg["train"], cfg["lora"]
        ckpt_kwargs = {"use_reentrant": False}
        if cfg["model"]["quantization"] == "4bit":
            self.model = prepare_model_for_kbit_training(
                self.model,
                use_gradient_checkpointing=t["gradient_checkpointing"],
                gradient_checkpointing_kwargs=ckpt_kwargs,
            )
        elif t["gradient_checkpointing"]:
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=ckpt_kwargs)
        self.model.config.use_cache = False
        self.model = get_peft_model(self.model, LoraConfig(
            r=lora["r"],
            lora_alpha=lora["alpha"],
            lora_dropout=lora["dropout"],
            target_modules=lora["target_modules"],
            task_type="CAUSAL_LM",
        ))
        for p in self.model.parameters():
            if p.requires_grad:
                p.data = p.data.float()  # fp16 mixed precision needs fp32 trainable (LoRA) weights
        self.model.print_trainable_parameters()

        data = self.sft_items(pairs)
        # Validation loss on a fixed sample (same pairs in every run, so loss curves are comparable)
        pick = np.random.default_rng(0).choice(len(val_pairs), size=min(t["val_pairs"], len(val_pairs)), replace=False)
        val_data = self.sft_items([val_pairs[i] for i in pick])

        args = TrainingArguments(
            **common_args(run_dir, t, seed, len(data)),
            save_strategy="steps",
            save_steps=t["save_steps"],
            eval_strategy="steps",
            eval_steps=t["eval_steps"],
            per_device_eval_batch_size=t["batch_size"],
            prediction_loss_only=True,  # loss only; keeping full-vocabulary logits would fill GPU memory
            remove_unused_columns=False,
        )
        trainer = Trainer(
            model=self.model, args=args, train_dataset=data, eval_dataset=val_data, data_collator=self.collate
        )
        out = trainer.train(resume_from_checkpoint=last_checkpoint(run_dir))
        trainer.evaluate()  # final validation-loss point at the last step
        self.model.save_pretrained(run_dir / "adapter")
        remove_checkpoints(run_dir)
        self.model.eval()
        return {"log_history": trainer.state.log_history, "train_metrics": out.metrics, "saved": run_dir / "adapter"}

    def sft_items(self, pairs):
        """Prompt + label tokens; the loss is only on the label tokens (the prompt is masked with -100)."""
        items = []
        for p in pairs:
            prompt = self.prompt_ids(p["text"], p["aspect"])
            target = self.tok.encode(p["label"], add_special_tokens=False)
            items.append({"input_ids": prompt + target, "labels": [-100] * len(prompt) + target})
        return items

    @torch.no_grad()
    def predict(self, texts, aspect, batch_size):
        self.model.eval()
        self.tok.padding_side = "left"  # the last position must be the real last token
        preds = []
        for i in range(0, len(texts), batch_size):
            ids = [self.prompt_ids(text, aspect) for text in texts[i : i + batch_size]]
            batch = self.tok.pad({"input_ids": ids}, return_tensors="pt").to(self.model.device)
            self.tokens_seen += int(batch["attention_mask"].sum())
            logits = self.model(**batch, logits_to_keep=1).logits[:, -1, self.label_ids]
            preds += logits.argmax(-1).tolist()
        return [LABELS[i] for i in preds]


class Encoder:
    """Sequence classifier on (aspect name, review) pairs, 3 classes. Best epoch on val is kept."""

    def __init__(self, cfg):
        m = cfg["model"]
        hf_id = m["nli_hf_id"] if cfg["run"]["mode"] == "few_shot" else m["hf_id"]
        self.tok = AutoTokenizer.from_pretrained(hf_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            hf_id,
            num_labels=3,
            id2label=dict(enumerate(LABELS)),
            label2id=LABEL_TO_ID,
            ignore_mismatched_sizes=True,  # NLI checkpoints come with their own head
        ).cuda()
        self.max_length = cfg["train"]["max_length"]
        self.tokens_seen = 0

    def tokenize(self, aspects, texts, **kwargs):
        return self.tok(
            [ASPECTS[a] for a in aspects], texts, truncation="only_second", max_length=self.max_length, **kwargs
        )

    def encode(self, pairs):
        enc = self.tokenize([p["aspect"] for p in pairs], [p["text"] for p in pairs])
        return [
            {**{key: enc[key][i] for key in enc.keys()}, "labels": LABEL_TO_ID[p["label"]]}
            for i, p in enumerate(pairs)
        ]

    def train(self, pairs, val_pairs, run_dir, cfg, seed):
        t = cfg["train"]
        args = TrainingArguments(
            **common_args(run_dir, t, seed, len(pairs)),
            per_device_eval_batch_size=64,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
        )
        trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=self.encode(pairs),
            eval_dataset=self.encode(val_pairs),
            data_collator=DataCollatorWithPadding(self.tok),
            # ponytail: epoch selection uses macro-F1 over all aspects pooled, not the per-aspect average
            compute_metrics=lambda p: {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")},
        )
        out = trainer.train(resume_from_checkpoint=last_checkpoint(run_dir))
        trainer.save_model(str(run_dir / "model"))
        remove_checkpoints(run_dir)
        self.model = trainer.model.eval()
        return {"log_history": trainer.state.log_history, "train_metrics": out.metrics, "saved": run_dir / "model"}

    @torch.no_grad()
    def predict(self, texts, aspect, batch_size):
        self.model.eval()
        preds = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i : i + batch_size]
            batch = self.tokenize([aspect] * len(chunk), chunk, padding=True, return_tensors="pt").to(self.model.device)
            self.tokens_seen += int(batch["attention_mask"].sum())
            with torch.autocast("cuda", dtype=torch.float16):
                preds += self.model(**batch).logits.argmax(-1).tolist()
        return [LABELS[i] for i in preds]


class NLIZeroShot:
    """Zero-shot encoder: for each label, ask the NLI model whether the review entails
    "The employee is <satisfied|neutral|dissatisfied> with the <aspect>." and pick the most entailed one."""

    VERBS = {"Low": "dissatisfied", "Medium": "neutral", "High": "satisfied"}

    def __init__(self, cfg):
        hf_id = cfg["model"]["nli_hf_id"]
        self.tok = AutoTokenizer.from_pretrained(hf_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(hf_id).cuda().eval()
        self.entail_id = next(i for i, name in self.model.config.id2label.items() if name.lower().startswith("entail"))
        self.max_length = cfg["train"]["max_length"]
        self.tokens_seen = 0

    @torch.no_grad()
    def predict(self, texts, aspect, batch_size):
        hypotheses = [f"The employee is {self.VERBS[label]} with the {ASPECTS[aspect]}." for label in LABELS]
        preds = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i : i + batch_size]
            batch = self.tok(
                [text for text in chunk for _ in hypotheses],  # each review once per hypothesis
                hypotheses * len(chunk),
                truncation="only_first",
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            ).to(self.model.device)
            self.tokens_seen += int(batch["attention_mask"].sum())
            with torch.autocast("cuda", dtype=torch.float16):
                entail = self.model(**batch).logits.float().softmax(-1)[:, self.entail_id]
            preds += entail.view(len(chunk), len(hypotheses)).argmax(-1).tolist()
        return [LABELS[i] for i in preds]

## 5. Evaluation

In [ ]:
AVERAGES = ["macro", "weighted"]


def evaluate(model, cfg, run_dir):
    """Macro F1, weighted F1 and accuracy per aspect + their average over the aspects, on every eval set.
    Saves predictions (for confusion matrices and bootstrap CIs) and times each set.
    Returns (scores, timing). One "review" = all 3 aspects predicted."""
    scores, timing = {}, {}
    for name in cfg["eval"]["sets"]:
        df = load_split(name)
        texts = df["text"].tolist()
        preds = df[["id", *ASPECTS]].copy()
        scores[name] = {}
        tokens_before = model.tokens_seen
        torch.cuda.synchronize()
        start = time.perf_counter()
        bs = cfg["eval"]["batch_size"]
        with tqdm(total=len(texts) * len(ASPECTS), desc=f"  {name} ({len(texts):,} reviews x 3 aspects)", unit="prompt") as bar:
            for aspect in ASPECTS:
                out = []
                for i in range(0, len(texts), bs):
                    out += model.predict(texts[i : i + bs], aspect, bs)
                    bar.update(len(texts[i : i + bs]))
                    bar.set_postfix(aspect=aspect, refresh=False)
                preds[aspect + "_pred"] = out
        torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        for aspect in ASPECTS:
            scores[name][aspect] = {
                avg: f1_score(df[aspect], preds[aspect + "_pred"], labels=LABELS, average=avg) for avg in AVERAGES
            }
            scores[name][aspect]["accuracy"] = float((df[aspect] == preds[aspect + "_pred"]).mean())
        scores[name]["avg"] = {m: float(np.mean([scores[name][a][m] for a in ASPECTS])) for m in scores[name][aspect]}
        timing[name] = {
            "reviews": len(df),
            "seconds": round(seconds, 2),
            "reviews_per_s": round(len(df) / seconds, 2),
            "ms_per_review": round(1000 * seconds / len(df), 2),
            "input_tokens_per_s": round((model.tokens_seen - tokens_before) / seconds),
        }
        preds.to_csv(run_dir / f"pred_{name}.csv", index=False)
        print(
            f"  {name:9s} macro-F1 = {scores[name]['avg']['macro']:.4f}   weighted-F1 = {scores[name]['avg']['weighted']:.4f}"
            f"   accuracy = {scores[name]['avg']['accuracy']:.4f}"
            f"   {timing[name]['reviews_per_s']:.1f} reviews/s"
        )
    return scores, timing

## 6. Run logging: model size, speed, plots

Saved in every run folder, for the presentation notebook:

| File | Contents |
|---|---|
| \`log_history.csv\` | training loss every 50 steps, learning rate, validation loss (LLM: every \`eval_steps\` on 500 fixed val pairs; encoders: every epoch, plus val macro-F1) |
| \`speed.json\` | training time and samples/s, inference reviews/s, ms per review and input tokens/s per eval set, peak GPU memory for training and for inference, GPU-hours |
| \`model_info.json\` | total and trainable parameters, size of the saved adapter / model in MB |
| \`loss.png\`, \`confusion_T1.png\` | quick sanity plots, also shown below each run |

Speed is measured on the timed eval sets after the first set (\`val\`) has warmed the GPU up; \`T1\` is the reference number.
Our models score one label token instead of generating text, so the speed metrics are **reviews per second** and
**input tokens per second** (how fast the prompt is read), not generated tokens per second.

In [ ]:
import json

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Light-mode chart tokens (slides are white). Series 1-2 of the categorical palette, blue sequential ramp.
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = {"train": "#2a78d6", "validation": "#eb6834"}
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 11, "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110, "savefig.dpi": 200,
})


def count_params(model):
    """(total, requires_grad) parameter counts. bitsandbytes packs two 4-bit weights per stored element."""
    total = trainable = 0
    for p in model.parameters():
        n = p.numel() * (2 if p.__class__.__name__ == "Params4bit" else 1)
        total += n
        trainable += n if p.requires_grad else 0
    return total, trainable


def dir_mb(path):
    return round(sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6, 1)


def plot_training(log, run_id, run_dir):
    """Train vs validation loss (one loss axis); encoders also get validation macro-F1 per epoch."""
    log = pd.DataFrame(log)
    if "loss" not in log:
        return
    train = log.dropna(subset=["loss"])
    val = log.dropna(subset=["eval_loss"]) if "eval_loss" in log else log.iloc[0:0]
    has_f1 = "eval_macro_f1" in log
    fig, axes = plt.subplots(1, 2 if has_f1 else 1, figsize=(11 if has_f1 else 6.5, 3.6), squeeze=False)
    ax = axes[0][0]
    ax.plot(train["step"], train["loss"], color=SERIES["train"], label="train")
    if len(val):
        ax.plot(val["step"], val["eval_loss"], color=SERIES["validation"], marker="o", markersize=6, label="validation")
    ax.set(title="Loss", xlabel="training step", ylabel="cross-entropy loss")
    ax.set_ylim(bottom=0)
    ax.legend()
    if has_f1:
        f1 = log.dropna(subset=["eval_macro_f1"])
        ax = axes[0][1]
        ax.plot(f1["epoch"], f1["eval_macro_f1"], color=SERIES["train"], marker="o", markersize=6)
        ax.set(title="Validation macro-F1 (all aspects pooled)", xlabel="epoch", ylabel="macro-F1")
        ax.set_xticks(f1["epoch"])
    fig.suptitle(run_id, color=INK_2, fontsize=10)
    fig.tight_layout()
    fig.savefig(run_dir / "loss.png")
    plt.show()


def plot_confusion(run_id, run_dir, test_set="T1"):
    """Row-normalised confusion matrix per aspect: share of each true class predicted as each class."""
    preds = pd.read_csv(run_dir / f"pred_{test_set}.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
    for ax, aspect in zip(axes, ASPECTS):
        true = preds[aspect].map(LABEL_TO_ID).to_numpy()
        pred = preds[aspect + "_pred"].map(LABEL_TO_ID).to_numpy()
        counts = np.zeros((3, 3), dtype=int)
        np.add.at(counts, (true, pred), 1)
        share = counts / counts.sum(axis=1, keepdims=True).clip(min=1)
        ax.imshow(share, cmap=BLUES, vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{share[i, j]:.0%}\n{counts[i, j]}", ha="center", va="center", fontsize=9,
                        color="white" if share[i, j] > 0.55 else INK)
        ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
               xlabel="predicted", ylabel="true", title=ASPECTS[aspect])
        ax.grid(False)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.suptitle(f"{run_id}: {test_set} confusion (% of each true class)", color=INK_2, fontsize=10)
    fig.tight_layout()
    fig.savefig(run_dir / f"confusion_{test_set}.png")
    plt.show()

## 7. Run

Expands the `sweep` (if any) into individual runs, then for each run: trains (if the mode trains), evaluates, and saves.
Run ids: `qwen3-1.7b_C2_full_s42` (finetune), `qwen3-1.7b_fewshot_k2_full_s42`, `qwen3-1.7b_zeroshot`,
plus any other swept values, e.g. `deberta-v3-base_C1_tune_s42_lr3e-05`.

In [ ]:
MODES = ("finetune", "zero_shot", "few_shot")


def expand_runs(cfg):
    """Yield (run_cfg, swept_values) for every combination in cfg["sweep"]."""
    sweep = cfg.get("sweep") or {}
    keys = list(sweep)
    for values in itertools.product(*(sweep[k] for k in keys)):
        c = copy.deepcopy(cfg)
        swept = dict(zip(keys, values))
        for key, value in swept.items():
            section = next((s for s in ("run", "train", "lora") if key in c.get(s, {})), None)
            assert section, f"sweep key '{key}' is not in run, train or lora"
            c[section][key] = value
        assert c["run"]["mode"] in MODES, f"run.mode must be one of {MODES}"
        if c["run"]["mode"] == "few_shot":  # encoder few-shot training settings; swept values still win
            c["train"].update({**c.get("few_shot", {}), **{k: v for k, v in swept.items() if k in c["train"]}})
        yield c, swept


def make_run_id(c, swept):
    r, name = c["run"], c["model"]["name"]
    if r["mode"] == "zero_shot":
        run_id = f"{name}_zeroshot"
    elif r["mode"] == "few_shot":
        run_id = f"{name}_fewshot_k{r['k_shot']}_{r['scale']}_s{r['seed']}"
    else:
        run_id = f"{name}_{r['condition']}_{r['scale']}_s{r['seed']}"
    for key, value in swept.items():
        if key not in ("mode", "condition", "scale", "seed", "k_shot"):
            run_id += f"_{key}{value:g}" if isinstance(value, float) else f"_{key}{value}"
    return run_id


def result_rows(c, run_id, scores, trained, speed, info):
    r = c["run"]
    llm_finetune = c["model"]["type"] == "llm" and r["mode"] == "finetune"
    return [
        {
            "run_id": run_id,
            "person": c.get("person", ""),
            "model": c["model"]["name"],
            "mode": r["mode"],
            "condition": r["condition"] if r["mode"] == "finetune" else "",
            "scale": r["scale"] if r["mode"] != "zero_shot" else "",
            "k_shot": r["k_shot"] if r["mode"] == "few_shot" else "",
            "seed": r["seed"],
            "lr": float(c["train"]["lr"]) if trained else "",
            "lora_r": c["lora"]["r"] if llm_finetune else "",
            "test_set": test_set,
            "aspect": aspect,
            "macro_f1": round(score["macro"], 4),
            "weighted_f1": round(score["weighted"], 4),
            "accuracy": round(score["accuracy"], 4),
            "train_min": speed["train_min"],
            "peak_train_gb": speed["peak_train_gb"],
            "peak_infer_gb": speed["peak_infer_gb"],
            "reviews_per_s": speed["eval"].get(test_set, {}).get("reviews_per_s", ""),
            "trainable_params": info["trainable_params"],
            "notes": "",
        }
        for test_set, by_aspect in scores.items()
        for aspect, score in by_aspect.items()
    ]


def build_model(c):
    if c["model"]["type"] == "llm":
        return LLM(c)
    return NLIZeroShot(c) if c["run"]["mode"] == "zero_shot" else Encoder(c)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# Plan: every run of the sweep (unique run ids, in order), and which are already done
PLAN = {}
for c, swept in expand_runs(CFG):
    PLAN.setdefault(make_run_id(c, swept), c)
todo = [run_id for run_id in PLAN if not (RUNS_DIR / run_id / "metrics.csv").exists()]
print(f"{len(PLAN)} runs for {CONFIG_NAME}, {len(todo)} to do:")
for run_id in PLAN:
    print(f"  {'todo' if run_id in todo else 'done'}  {run_id}")

for n, run_id in enumerate(todo, 1):
    c = PLAN[run_id]
    run = c["run"]
    run_dir = RUNS_DIR / run_id
    run_start = time.time()

    print(f"\n{'=' * 70}\n[{n}/{len(todo)}] {run_id}\n{'=' * 70}")
    resuming = last_checkpoint(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.yaml").write_text(yaml.safe_dump(c, sort_keys=False))
    set_seed(run["seed"])
    torch.cuda.reset_peak_memory_stats()

    log(f"loading model ({run['mode']}) ...")
    model = build_model(c)
    log(f"model loaded, GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    llm = c["model"]["type"] == "llm"
    trained = run["mode"] == "finetune" or (run["mode"] == "few_shot" and not llm)
    result = {}
    start = time.time()
    if run["mode"] == "few_shot" and llm:
        model.set_shots(training_pairs(run), run["seed"])
        log(f"in-context examples per aspect: {run['k_shot'] * len(LABELS)} (no training)")
    elif trained:
        pairs = training_pairs(run)
        t = c["train"]
        steps = math.ceil(len(pairs) / (t["batch_size"] * t.get("grad_accum", 1))) * t["epochs"]
        log(f"training pairs: {len(pairs):,} | {t['epochs']} epoch(s), {steps:,} optimizer steps, lr {float(t['lr']):g}"
            + (" | resuming from checkpoint" if resuming else ""))
        result = model.train(pairs, to_pairs(load_split("val")), run_dir, c, run["seed"])
        pd.DataFrame(result["log_history"]).to_csv(run_dir / "log_history.csv", index=False)
        log(f"training done in {(time.time() - start) / 60:.1f} min")
    train_s = time.time() - start if trained else 0.0
    peak_train_gb = torch.cuda.max_memory_allocated() / 1e9 if trained else None

    torch.cuda.reset_peak_memory_stats()
    log(f"evaluating on {len(c['eval']['sets'])} sets: {', '.join(c['eval']['sets'])}")
    scores, timing = evaluate(model, c, run_dir)
    total, trainable = count_params(model.model)
    info = {
        "hf_id": getattr(model.model, "name_or_path", "") or c["model"]["hf_id"],
        "total_params": total,
        "trainable_params": trainable if trained else 0,
        "trainable_pct": round(100 * trainable / total, 3) if trained else 0.0,
        "saved_mb": dir_mb(result["saved"]) if trained else None,
    }
    train_metrics = result.get("train_metrics", {})
    speed = {
        "gpu": torch.cuda.get_device_name(0),
        "train_min": round(train_s / 60, 1),
        "train_samples_per_s": train_metrics.get("train_samples_per_second"),
        "train_steps_per_s": train_metrics.get("train_steps_per_second"),
        "peak_train_gb": round(peak_train_gb, 2) if trained else None,
        "peak_infer_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "eval_batch_size": c["eval"]["batch_size"],
        "eval": timing,
        "gpu_hours": round((train_s + sum(t["seconds"] for t in timing.values())) / 3600, 3),
    }
    (run_dir / "speed.json").write_text(json.dumps(speed, indent=2))
    (run_dir / "model_info.json").write_text(json.dumps(info, indent=2))
    t1 = timing.get("T1", {})
    print(f"  trainable params {info['trainable_params']:,} of {total:,} ({info['trainable_pct']}%)"
          f" | T1 {t1.get('reviews_per_s')} reviews/s, {t1.get('input_tokens_per_s')} input tokens/s"
          f" | peak GPU train {speed['peak_train_gb']} GB, inference {speed['peak_infer_gb']} GB")
    if trained:
        plot_training(result["log_history"], run_id, run_dir)
    if "T1" in timing:
        plot_confusion(run_id, run_dir)

    rows = result_rows(c, run_id, scores, trained, speed, info)
    # metrics.csv per run is the source of truth (and marks the run as done); results.csv is the shared summary
    pd.DataFrame(rows).to_csv(run_dir / "metrics.csv", index=False)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    log(f"run {run_id} done in {(time.time() - run_start) / 60:.1f} min, saved to {run_dir}")

## 8. Results for this model (average over the 3 aspects)

In [ ]:
results = pd.read_csv(RESULTS_CSV)
results = results[(results["model"] == CFG["model"]["name"]) & (results["aspect"] == "avg")]
for metric in ["macro_f1", "weighted_f1", "accuracy"]:
    print(metric)
    display(results.pivot_table(index="run_id", columns="test_set", values=metric).round(4))

print("speed and cost (T1)")
speed_cols = ["run_id", "mode", "train_min", "peak_train_gb", "peak_infer_gb", "reviews_per_s", "trainable_params"]
display(results[results["test_set"] == "T1"][speed_cols].drop_duplicates("run_id").set_index("run_id"))